# Embedding Tuning - Turkish Legal QA (A100)

**Proje:** CENG493 Turkish Legal RAG  
**Adim:** 2/5 - Embedding Tuning  
**GPU:** A100 (40GB)  
**Onceki:** Baseline RAG (Recall@5=78.7%, F1=0.272)

**Proje gereksinimleri:**
1. Domain adaptation
2. Contrastive fine-tuning
3. Hard negative mining
4. Hybrid retrieval (BM25 + Dense)

**Veri:** Kaggle `turkish_law_dataset.csv` — soru + satirdaki `context` ile korpus chunk indeksi hizalanir; hard negative retrieval ile secilir (HF CSV kullanilmaz).

**ONEMLI — `chunks.jsonl` kaynagi:** RAG korpusu **`scripts/prepare_corpus.py`** ile uretilmeli (cikti: `data/processed/chunks.jsonl`). Boylece her chunk metni = CSV `context` ile **bire bir** (normalize sonrasi); 02'de tam eslesme + guvenilir pozitif. **`chunk_corpus.py` sliding-window** ciktilari Kaggle `context` ile **eslesmez**; embedding triple ve RAG hatti kirilir.

**Iyilestirmeler (vs onceki deney):**
- Pozitif: **context -> chunk** tam metin eslemesi (`prepare_corpus` = ayni `normalize_whitespace`); gerekirse FAISS fallback
- batch_size: 4 -> 8 (7 in-batch negatif + 1 hard negatif)
- Hard negative mining (farkli baglam / `article_key` = `doc_id`)
- LR: 2e-5 -> 5e-6, Epoch: 3 -> 2
- Hybrid retrieval (RRF fusion) eklendi

In [ ]:
!pip install -q sentence-transformers faiss-cpu transformers accelerate bitsandbytes rank_bm25 snowballstemmer rouge-score nltk

import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import json
import re
import numpy as np
import pandas as pd
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/493_project")

import sys
sys.path.insert(0, str(PROJECT_DIR))
from evaluation.metrics_utils import (
    RAG_SYSTEM_PROMPT,
    format_context_blocks_for_llm,
    aggregate_qa_row,
    retrieval_hit,
)

chunks = []
with open(PROJECT_DIR / "data/processed/chunks.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            chunks.append(json.loads(line))
chunk_texts = [c["text"] for c in chunks]

with open(PROJECT_DIR / "evaluation/gold_test_set.json", "r", encoding="utf-8") as f:
    gold_questions = json.load(f)
answerable_qs = [q for q in gold_questions if q.get("answerable_from_corpus", True)]

with open(PROJECT_DIR / "evaluation/baseline_results.json", "r", encoding="utf-8") as f:
    baseline_results = json.load(f)

kg_raw = pd.read_csv(PROJECT_DIR / "data/raw/turkish_law_dataset.csv")

print(f"Chunks: {len(chunks)}")
print(f"Answerable Qs: {len(answerable_qs)}")
print(f"Kaggle raw rows: {len(kg_raw)}")
print(f"Baseline Recall@5: {baseline_results['metrics']['recall@5']:.4f}")

## 1. Training Data + Hard Negative Mining (Kaggle)

- **Pozitif:** Ayni satirdaki `context` metni, `prepare_corpus.py` ile ayni `normalize_whitespace` sonrasi chunk `text` ile eslestirilir (bire bir indeks).
- **Eslesmeyen satirlar:** Context embedding ile FAISS'ta en yakin chunk; kosinus >= **0.85** ise pozitif kabul.
- **Hard negative:** Sorunun base BGE embedding'i ile top-30 aday icinde, pozitiften **farkli `article_key`** olan ilk chunk.
- Gold test sorulari ve `Score < 8` satirlari elenir.

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss

embed_model = SentenceTransformer("BAAI/bge-m3", device="cuda")

# Hat A (uzun context): 128 OOM; 32 genelde A100/L4 icin makul. Hâlâ OOM -> 16.
CHUNK_ENCODE_BATCH = 32

print("Encoding chunks...")
chunk_vecs = embed_model.encode(
    chunk_texts, batch_size=CHUNK_ENCODE_BATCH,
    show_progress_bar=True, normalize_embeddings=True,
)
chunk_vecs = np.array(chunk_vecs).astype("float32")

dim = chunk_vecs.shape[1]
base_index = faiss.IndexFlatIP(dim)
base_index.add(chunk_vecs)
print(f"Base index: {base_index.ntotal} vectors, dim={dim}")
print(f"VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB")

In [ ]:
from tqdm import tqdm

REQUIRED = ["soru", "cevap", "veri türü", "kaynak", "context", "Score"]
missing = [c for c in REQUIRED if c not in kg_raw.columns]
if missing:
    raise ValueError(f"Kaggle CSV eksik kolon: {missing}")

MIN_SCORE = 8
MIN_CONTEXT_LEN = 100
FALLBACK_SIM = 0.85
TOPK_HARDNEG = 30


def normalize_whitespace(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = text.replace("\ufeff", " ").replace("\xa0", " ")
    lines = [line.strip() for line in text.split("\n")]
    cleaned_lines, previous_blank = [], False
    for line in lines:
        is_blank = line == ""
        if is_blank and previous_blank:
            continue
        cleaned_lines.append(line)
        previous_blank = is_blank
    text = "\n".join(cleaned_lines).strip()
    text = re.sub(r"[ \t]+", " ", text)
    return text


kg = kg_raw.copy()
kg["Score"] = pd.to_numeric(kg["Score"], errors="coerce")
kg = kg[kg["Score"] >= MIN_SCORE].copy()
kg["soru"] = kg["soru"].astype(str).apply(normalize_whitespace)
kg["cevap"] = kg["cevap"].astype(str).apply(normalize_whitespace)
kg["context_clean"] = kg["context"].astype(str).apply(normalize_whitespace)
kg = kg[
    (kg["soru"].str.len() > 0)
    & (kg["cevap"].str.len() > 0)
    & (kg["context_clean"].str.len() >= MIN_CONTEXT_LEN)
].copy()

gold_q_set = {q["soru"].lower().strip() for q in gold_questions}
kg = kg[~kg["soru"].str.lower().str.strip().isin(gold_q_set)].copy()
kg = kg.drop_duplicates(keep="first").reset_index(drop=True)
print(f"Kaggle satirlari (filtre + gold cikarim + satir dedup): {len(kg)}")

context_to_idx: dict[str, int] = {}
for i, t in enumerate(chunk_texts):
    key = normalize_whitespace(t)
    if key not in context_to_idx:
        context_to_idx[key] = i

pos_indices: list[int] = []
exact_hits = 0
missing_for_fallback: list[tuple[int, str]] = []

for _, row in kg.iterrows():
    cn = row["context_clean"]
    pid = context_to_idx.get(cn)
    if pid is not None:
        pos_indices.append(pid)
        exact_hits += 1
    else:
        pos_indices.append(-1)
        missing_for_fallback.append((len(pos_indices) - 1, cn))

if missing_for_fallback:
    print(f"Context tam eslesmeyen satir: {len(missing_for_fallback)} (FAISS fallback)...")
    FB_BATCH = globals().get("CHUNK_ENCODE_BATCH", 32)
    for start in tqdm(range(0, len(missing_for_fallback), FB_BATCH), desc="Context fallback"):
        batch_slice = missing_for_fallback[start : start + FB_BATCH]
        texts = [t for _, t in batch_slice]
        qv = embed_model.encode(texts, normalize_embeddings=True, show_progress_bar=False)
        qv = np.array(qv).astype("float32")
        sc, ind = base_index.search(qv, 1)
        for bi, (row_i, _) in enumerate(batch_slice):
            if float(sc[bi][0]) >= FALLBACK_SIM:
                pos_indices[row_i] = int(ind[bi][0])
            else:
                pos_indices[row_i] = -1

keep = np.array([p >= 0 for p in pos_indices], dtype=bool)
kg = kg.iloc[np.where(keep)[0]].copy().reset_index(drop=True)
pos_indices = [pos_indices[i] for i in range(len(pos_indices)) if keep[i]]
print(
    f"Pozitif chunk atanmis ornek: {len(kg)} "
    f"(context-chunk tam esleme sayisi ilk geciste: {exact_hits})"
)

kg_questions = kg["soru"].tolist()
print(f"Kaggle sorulari encode + top-{TOPK_HARDNEG} arama...")
BATCH = 512
all_scores = []
all_indices = []
for i in tqdm(range(0, len(kg_questions), BATCH)):
    batch = kg_questions[i : i + BATCH]
    q_vecs = embed_model.encode(batch, normalize_embeddings=True)
    q_vecs = np.array(q_vecs).astype("float32")
    scores, indices = base_index.search(q_vecs, TOPK_HARDNEG)
    all_scores.append(scores)
    all_indices.append(indices)

all_scores = np.vstack(all_scores)
all_indices = np.vstack(all_indices)
print(f"Arama tamam: {all_scores.shape}")

In [ ]:
train_triplets = []
pairs_only = []

n_q = len(kg_questions)
assert len(pos_indices) == n_q and all_indices.shape[0] == n_q

for i in range(n_q):
    pos_idx = pos_indices[i]
    pos_article = chunks[pos_idx].get("article_key", "")
    hard_neg_idx = None

    for j in range(TOPK_HARDNEG):
        idx = int(all_indices[i][j])
        if idx == pos_idx:
            continue
        if chunks[idx].get("article_key", "") != pos_article:
            hard_neg_idx = idx
            break

    pairs_only.append({
        "query": kg_questions[i],
        "positive": chunk_texts[pos_idx],
    })
    if hard_neg_idx is not None:
        train_triplets.append({
            "query": kg_questions[i],
            "positive": chunk_texts[pos_idx],
            "negative": chunk_texts[hard_neg_idx],
        })

print(f"Pairs (Kaggle hizali pozitif): {len(pairs_only)}")
print(f"Triplets (hard negative bulundu): {len(train_triplets)}")

save_path = PROJECT_DIR / "data/processed/embedding_train_triplets.json"
with open(save_path, "w", encoding="utf-8") as f:
    json.dump(train_triplets, f, ensure_ascii=False)
print(f"Saved -> {save_path}")

## 2. Contrastive Fine-tuning

**MultipleNegativesRankingLoss** ile fine-tune:  
- Hard negative'ler batch'e dahil  
- chunk encode batch=32 (uzun metin); fine-tune batch=6, max_seq_length=256  
- lr=5e-6, epochs=2 (catastrophic forgetting onlemi)

In [ ]:
import gc
import random
from sentence_transformers import InputExample, losses
from torch.utils.data import DataLoader

del chunk_vecs, base_index, all_scores, all_indices
gc.collect()
torch.cuda.empty_cache()
print(f"VRAM after cleanup: {torch.cuda.memory_allocated()/1e9:.1f} GB")

random.seed(42)
random.shuffle(train_triplets)
MAX_TRAIN = min(len(train_triplets), 3000)
selected = train_triplets[:MAX_TRAIN]
print(f"Using {len(selected)} triplets for training")

embed_model.max_seq_length = 256

train_examples = [
    InputExample(texts=[t["query"], t["positive"], t["negative"]])
    for t in selected
]

BATCH_SIZE = 6  # 8 ile 4 arasi; OOM -> 4
EPOCHS = 2
LR = 5e-6

train_dataloader = DataLoader(
    train_examples, shuffle=True, batch_size=BATCH_SIZE, num_workers=0, pin_memory=False,
)
train_loss = losses.MultipleNegativesRankingLoss(embed_model)
WARMUP = int(len(train_dataloader) * EPOCHS * 0.2)

save_model_path = str(PROJECT_DIR / "models" / "bge-m3-legal-v2")

print(f"batch_size={BATCH_SIZE}, epochs={EPOCHS}, lr={LR}, warmup={WARMUP}")
print(f"Batches/epoch: {len(train_dataloader)}")
print(f"Save: {save_model_path}")

embed_model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=EPOCHS,
    warmup_steps=WARMUP,
    optimizer_params={"lr": LR},
    output_path=save_model_path,
    show_progress_bar=True,
    use_amp=True,
)

print(f"\nFine-tuning complete!")
print(f"VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB")

## 3. New FAISS Index + BM25 Hybrid Retrieval

Fine-tune edilmis model ile yeni FAISS index olusturup,  
BM25 index ile birlestirerek **Reciprocal Rank Fusion (RRF)** uyguluyoruz.

In [ ]:
torch.cuda.empty_cache()

tuned_model = embed_model

print("Re-encoding chunks with tuned model...")
_ceb = globals().get("CHUNK_ENCODE_BATCH", 32)
tuned_vecs = tuned_model.encode(
    chunk_texts, batch_size=_ceb,
    show_progress_bar=True, normalize_embeddings=True,
)
tuned_vecs = np.array(tuned_vecs).astype("float32")

tuned_index = faiss.IndexFlatIP(tuned_vecs.shape[1])
tuned_index.add(tuned_vecs)
print(f"Tuned FAISS index: {tuned_index.ntotal} vectors")

tuned_save_dir = PROJECT_DIR / "models" / "embedding_tuned_v2"
tuned_save_dir.mkdir(parents=True, exist_ok=True)
faiss.write_index(tuned_index, str(tuned_save_dir / "faiss.index"))
np.save(str(tuned_save_dir / "dense_vecs.npy"), tuned_vecs)
print(f"Saved -> {tuned_save_dir}")

In [ ]:
import snowballstemmer
from rank_bm25 import BM25Okapi

tr_stemmer = snowballstemmer.stemmer('turkish')

TURKISH_STOPWORDS = {
    "bir", "bu", "de", "da", "ve", "ile", "için", "olan", "olarak",
    "veya", "ya", "hem", "ne", "her", "o", "şu", "ki", "mi", "mu",
    "mı", "mü", "ise", "den", "dan", "ten", "tan", "dir", "dır",
    "dür", "dur", "tir", "tır", "tur", "tür", "gibi", "kadar",
    "sonra", "önce", "üzere", "göre", "ait", "dair", "en", "çok",
    "var", "yok", "olan", "olup", "ancak", "fakat", "ama",
}

def tokenize_tr(text: str) -> list:
    tokens = re.findall(r'\w+', text.lower())
    tokens = [t for t in tokens if t not in TURKISH_STOPWORDS and len(t) > 1]
    return tr_stemmer.stemWords(tokens)

print("Building BM25 index with Turkish stemming...")
tokenized_chunks = [tokenize_tr(t) for t in chunk_texts]
bm25 = BM25Okapi(tokenized_chunks)
print(f"BM25 index ready: {len(tokenized_chunks)} documents")

test_tokens = tokenize_tr("Mahkeme kararlarına itiraz süresi nedir?")
print(f"Stemming test: {test_tokens}")


def _make_result(idx: int, rank: int, score: float) -> dict:
    c = chunks[idx]
    return {
        "rank": rank, "score": score,
        "chunk_id": c["chunk_id"], "source": c["source"],
        "article_title": c.get("article_title", ""),
        "article_key": c.get("article_key", ""),
        "text": c["text"],
    }


def retrieve_dense(query: str, top_k: int = 10) -> list:
    q_vec = tuned_model.encode([query], normalize_embeddings=True)
    q_vec = np.array(q_vec).astype("float32")
    scores, indices = tuned_index.search(q_vec, top_k)
    return [_make_result(int(idx), r+1, float(s))
            for r, (s, idx) in enumerate(zip(scores[0], indices[0]))]


def retrieve_hybrid(query: str, top_k: int = 10, alpha: float = 0.8) -> list:
    q_vec = tuned_model.encode([query], normalize_embeddings=True)
    q_vec = np.array(q_vec).astype("float32")

    dense_scores = np.dot(tuned_vecs, q_vec.T).flatten()
    bm25_scores = bm25.get_scores(tokenize_tr(query))

    d_min, d_max = dense_scores.min(), dense_scores.max()
    dense_norm = (dense_scores - d_min) / (d_max - d_min + 1e-9)

    b_max = bm25_scores.max()
    bm25_norm = bm25_scores / (b_max + 1e-9)

    combined = alpha * dense_norm + (1 - alpha) * bm25_norm
    top_indices = np.argsort(combined)[::-1][:top_k]

    return [_make_result(int(idx), r+1, float(combined[idx]))
            for r, idx in enumerate(top_indices)]


print("\nTest: 'Egemenlik kime aittir?'")
print("\n--- Dense ---")
for r in retrieve_dense("Egemenlik kime aittir?", 3):
    print(f"  R{r['rank']} ({r['score']:.4f}) [{r['source']} | {r['article_title']}]")
print("\n--- Hybrid (alpha=0.8 Dense + 0.2 BM25) ---")
for r in retrieve_hybrid("Egemenlik kime aittir?", 3):
    print(f"  R{r['rank']} ({r['score']:.4f}) [{r['source']} | {r['article_title']}]")

## 4. Evaluation

Uc konfigurasyon karsilastirilacak:
1. Baseline (base bge-m3, dense only)
2. Tuned Dense (fine-tuned bge-m3, dense only)
3. Tuned Hybrid (fine-tuned bge-m3, dense + BM25 RRF)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

LLM_NAME = "Qwen/Qwen2.5-7B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(LLM_NAME)
llm = AutoModelForCausalLM.from_pretrained(
    LLM_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)
print(f"LLM loaded | VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB")


def rag_answer(question: str, retrieved: list) -> str:
    context = format_context_blocks_for_llm(retrieved, max_chunks=5)
    messages = [
        {"role": "system", "content": RAG_SYSTEM_PROMPT},
        {"role": "user", "content": f"Baglam:\n{context}\n\nSoru: {question}"},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=4096).to(llm.device)
    with torch.no_grad():
        out = llm.generate(**inputs, max_new_tokens=384, do_sample=False)
    return tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

In [ ]:
import nltk

for _p in ("punkt", "punkt_tab"):
    try:
        nltk.download(_p, quiet=True)
    except Exception:
        pass

print("Eval: evaluation.metrics_utils")


In [ ]:
from tqdm import tqdm

TOP_K = 10

results_dense = []
results_hybrid = []

for q in tqdm(answerable_qs, desc="Evaluating"):
    ret_d = retrieve_dense(q["soru"], top_k=TOP_K)
    ret_h = retrieve_hybrid(q["soru"], top_k=TOP_K)

    rh_d = retrieval_hit(ret_d, q)
    rh_h = retrieval_hit(ret_h, q)

    ans_h = rag_answer(q["soru"], ret_h)
    qa_h = aggregate_qa_row(ans_h, q["cevap"], ret_h, q)
    pred_body = qa_h.pop("pred_body")

    base = {
        "id": q["id"], "soru": q["soru"],
        "gold_cevap": q["cevap"], "kaynak": q["kaynak"],
        "difficulty": q["difficulty"], "type": q["type"],
    }

    z = dict.fromkeys(["f1", "em", "bleu", "rouge1", "rouge2", "rougeL", "faithfulness_token_recall",
                       "citation_precision_retrieved", "citation_recall_gold", "num_citations"], 0.0)
    results_dense.append({**base, **rh_d, **z})

    results_hybrid.append({**base, **rh_h, "pred_cevap": ans_h, "pred_body": pred_body, **qa_h})

print(f"\nDone: {len(results_hybrid)} questions evaluated")

In [ ]:
df_d = pd.DataFrame(results_dense)
df_h = pd.DataFrame(results_hybrid)
bl = baseline_results["metrics"]

print("=" * 70)
print("  EMBEDDING TUNING SONUCLARI")
print("=" * 70)

print(f"\n{'Metric':<12} {'Baseline':>10} {'Tuned Dense':>12} {'Tuned Hybrid':>13}")
print("-" * 49)
for k in [1, 3, 5, 10]:
    b = bl[f"recall@{k}"]
    d = df_d[f"hit@{k}"].mean()
    h = df_h[f"hit@{k}"].mean()
    print(f"Recall@{k:<2d}    {b:>9.4f}  {d:>11.4f}  {h:>12.4f}")

b_mrr = bl["mrr"]
d_mrr = df_d["mrr"].mean()
h_mrr = df_h["mrr"].mean()
print(f"{'MRR':<12} {b_mrr:>9.4f}  {d_mrr:>11.4f}  {h_mrr:>12.4f}")

b_ndcg = bl.get("ndcg@10", float("nan"))
d_ndcg = df_d["ndcg@10"].mean()
h_ndcg = df_h["ndcg@10"].mean()
print(f"{'nDCG@10':<12} {b_ndcg:>9.4f}  {d_ndcg:>11.4f}  {h_ndcg:>12.4f}")

print(f"\n{'Metric':<14} {'Baseline':>10} {'Tuned Hybrid':>13} {'Delta':>8}")
print("-" * 48)
h_f1 = df_h["f1"].mean()
h_em = df_h["em"].mean()
print(f"{'F1':<14} {bl['f1']:>9.4f}  {h_f1:>12.4f}  {h_f1-bl['f1']:>+7.4f}")
print(f"{'EM':<14} {bl['em']:>9.4f}  {h_em:>12.4f}  {h_em-bl['em']:>+7.4f}")
for name, col in [("BLEU", "bleu"), ("ROUGE-1", "rouge1"), ("ROUGE-2", "rouge2"), ("ROUGE-L", "rougeL"),
                  ("Faithful.", "faithfulness_token_recall"), ("CitePrec", "citation_precision_retrieved")]:
    bv = bl.get(col, float("nan"))
    hv = df_h[col].mean()
    dlt = hv - bv if bv == bv else float("nan")
    print(f"{name:<14} {bv:>9.4f}  {hv:>12.4f}  {dlt:>+7.4f}" if bv == bv else f"{name:<14} {'(n/a)':>9}  {hv:>12.4f}")
_bcr = df_h.loc[df_h["citation_recall_gold"] >= 0, "citation_recall_gold"]
print(f"{'CiteRecGold':<14} {'(n/a)':>9}  {_bcr.mean():>12.4f}  (n={len(_bcr)})")

print("\n--- Difficulty (Tuned Hybrid) ---")
for d in ["easy", "medium", "hard", "very hard"]:
    s = df_h[df_h["difficulty"] == d]
    if len(s):
        print(f"  {d:10s}: R@5={s['hit@5'].mean():.3f}  F1={s['f1'].mean():.3f}  R-L={s['rougeL'].mean():.3f}  (n={len(s)})")

print("\n--- Question Type (Tuned Hybrid) ---")
for t in sorted(df_h["type"].unique()):
    s = df_h[df_h["type"] == t]
    print(f"  {t:15s}: R@5={s['hit@5'].mean():.3f}  F1={s['f1'].mean():.3f}  R-L={s['rougeL'].mean():.3f}  (n={len(s)})")

In [ ]:
_bcr = df_h.loc[df_h["citation_recall_gold"] >= 0, "citation_recall_gold"]

output = {
    "experiment": "embedding_tuning_v2",
    "embedding": "BAAI/bge-m3 (legal fine-tuned, hard negatives)",
    "llm": "Qwen/Qwen2.5-7B-Instruct-4bit",
    "retrieval": "hybrid_dense_bm25_rrf",
    "reranker": None,
    "training": {
        "triplets": len(selected),
        "triplet_source": "kaggle_turkish_law_dataset_csv",
        "positive_match": "context_normalize_then_chunk_index_or_faiss_fallback_cosine>=0.85",
        "hard_negative_topk": TOPK_HARDNEG,
        "batch_size": BATCH_SIZE,
        "epochs": EPOCHS,
        "lr": LR,
        "loss": "MultipleNegativesRankingLoss",
        "hard_negatives": True,
    },
    "num_questions": len(results_hybrid),
    "metrics_tuned_dense": {
        f"recall@{k}": float(df_d[f"hit@{k}"].mean()) for k in [1,3,5,10]
    } | {"mrr": float(d_mrr), "ndcg@10": float(d_ndcg)},
    "metrics_tuned_hybrid": {
        f"recall@{k}": float(df_h[f"hit@{k}"].mean()) for k in [1,3,5,10]
    } | {
        "mrr": float(h_mrr),
        "ndcg@10": float(h_ndcg),
        "f1": float(h_f1),
        "em": float(h_em),
        "bleu": float(df_h["bleu"].mean()),
        "rouge1": float(df_h["rouge1"].mean()),
        "rouge2": float(df_h["rouge2"].mean()),
        "rougeL": float(df_h["rougeL"].mean()),
        "faithfulness_token_recall": float(df_h["faithfulness_token_recall"].mean()),
        "citation_precision_retrieved": float(df_h["citation_precision_retrieved"].mean()),
        "citation_recall_gold_mean": float(_bcr.mean()) if len(_bcr) else None,
    },
    "baseline_metrics": bl,
    "per_question_hybrid": results_hybrid,
}

save_path = PROJECT_DIR / "evaluation" / "embedding_tuning_results.json"
with open(save_path, "w", encoding="utf-8") as f:
    json.dump(output, f, ensure_ascii=False, indent=2)

print(f"Results saved -> {save_path}")
print("\nSonraki adim: Reranker ekleme (bge-reranker-v2-m3)")